### Simulación de 4 cuerpos para el caso de un agujero negro.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter 

# =============================================================================
# 1. Parámetros de la simulación
# =============================================================================
G = 0.004498         # Cte de gravitación universal
epsilon = 0.05       # Coeficiente anti-choque
dt = 0.0005          # Distancia entre los pasos
n = 15000            # Cantidad de iteraciones
R_col = 1.0          # DISTANCIA DE COLISIÓN (en parsecs).
M_limite_fusion = 5 * 10**7 # MASA LÍMITE: Si la masa supera esto, TRAGA. Si no, REBOTA.

# =============================================================================
# 2. Condiciones iniciales (Modificadas para mostrar rebote y luego fusión)
# =============================================================================
masas = np.array([10**8, 10**6, 10**6, 5*10**8]) 
N = len(masas)

# Ponemos a los dos planetas en curso de colisión frontal arriba de la estrella
posiciones = np.array([
    [ 0.0,   0.0,  0.0],  # Estrella Local en el centro
    [ 5.0,  10.0,  2.0],  # Planeta 1 (con inclinación en Z)
    [-5.0,  10.0, -2.0],  # Planeta 2 (con inclinación en Z)
    [ 0.0, -30.0,-15.0]   # Invasor Masivo acercándose desde abajo en diagonal
], dtype=float)

velocidades = np.array([
    [   0.0,   0.0,  0.0],
    [-100.0,   0.0, -5.0], # Planeta 1 vuela hacia la izquierda y abajo
    [ 100.0,   0.0,  5.0], # Planeta 2 vuela hacia la derecha y arriba
    [   0.0, 150.0, 75.0]  # Invasor sube a romper madres en 3D
], dtype=float)

colores = ['darkorange', 'cyan', 'lime', 'black']
etiquetas = ['Estrella Local', 'Planeta 1', 'Planeta 2', 'Invasor Masivo']
limite_ejes = 35

esta_vivo = np.ones(N, dtype=bool)

historial_x = np.zeros((n, N))
historial_y = np.zeros((n, N))
historial_z = np.zeros((n, N))

# =============================================================================
# 3. Motor Físico: Cálculo de la aceleración
# =============================================================================
def calcular_aceleraciones(pos, m, vivos):
    aceleraciones = np.zeros((N, 3))
    for i in range(N):
        if not vivos[i]: continue 
        for j in range(N):
            if i != j and vivos[j]: 
                dr = pos[j] - pos[i]
                norm = dr[0]**2 + dr[1]**2 + dr[2]**2           
                normepsilon = norm + epsilon**2                 
                aceleraciones[i] += G * m[j] * dr / (normepsilon ** 1.5)  
    return aceleraciones

# =============================================================================
# 4. Integración Numérica con Lógica Híbrida de Colisión
# =============================================================================
iteracion = 0
while iteracion < n:
    for i in range(N):  
        if esta_vivo[i]:
            historial_x[iteracion, i] = posiciones[i, 0]
            historial_y[iteracion, i] = posiciones[i, 1]
            historial_z[iteracion, i] = posiciones[i, 2]
        else:
            historial_x[iteracion, i] = np.nan
            historial_y[iteracion, i] = np.nan
            historial_z[iteracion, i] = np.nan

    acc = calcular_aceleraciones(posiciones, masas, esta_vivo)    

    # 1. Integración de movimiento
    for i in range(N):
        if not esta_vivo[i]: continue
        posiciones[i] += velocidades[i] * dt + 0.5 * acc[i] * (dt**2) 
        velocidades[i] += acc[i] * dt  
        
    # 2. Detector de Colisiones (Híbrido: Rebote vs Fusión)
    for i in range(N):
        if not esta_vivo[i]: continue
        for j in range(i + 1, N):
            if not esta_vivo[j]: continue
            
            dr = posiciones[j] - posiciones[i]
            distancia = np.sqrt(dr[0]**2 + dr[1]**2 + dr[2]**2)
            
            if distancia < R_col:
                # Verificamos si alguno de los dos es lo suficientemente masivo para tragar
                if masas[i] >= M_limite_fusion or masas[j] >= M_limite_fusion:
                    print(f"[{iteracion}] Colisión INELÁSTICA (Fusión). '{etiquetas[i]}' y '{etiquetas[j]}'")
                    
                    m_total = masas[i] + masas[j]
                    r_cm = (masas[i]*posiciones[i] + masas[j]*posiciones[j]) / m_total
                    v_cm = (masas[i]*velocidades[i] + masas[j]*velocidades[j]) / m_total
                    
                    if masas[i] >= masas[j]:
                        masas[i], posiciones[i], velocidades[i] = m_total, r_cm, v_cm
                        esta_vivo[j] = False  
                        masas[j] = 0.0
                    else:
                        masas[j], posiciones[j], velocidades[j] = m_total, r_cm, v_cm
                        esta_vivo[i] = False  
                        masas[i] = 0.0
                
                else:
                    # Si ambos son "pequeños", chocan elásticamente (Rebote espacial)
                    n_vec = dr / distancia  # Vector normal unitario
                    v_rel = velocidades[j] - velocidades[i]
                    v_rn = np.dot(v_rel, n_vec)
                    
                    # Solo chocan si se están acercando
                    if v_rn < 0:
                        print(f"[{iteracion}] Colisión ELÁSTICA (Rebote). '{etiquetas[i]}' y '{etiquetas[j]}'")
                        
                        # Ecuación del impulso escalar (e = 1.0 para choque perfectamente elástico)
                        J = -2.0 * v_rn / (1.0/masas[i] + 1.0/masas[j])
                        
                        # Modificamos las órbitas (velocidades)
                        velocidades[i] -= (J / masas[i]) * n_vec
                        velocidades[j] += (J / masas[j]) * n_vec
                        
                        # Truco de físico computacional (Anti-pegado):
                        # Los separamos a huevo una fracción minúscula para que el integrador no los atrape en el siguiente dt
                        overlap = R_col - distancia
                        mtot = masas[i] + masas[j]
                        posiciones[i] -= n_vec * (overlap * masas[j] / mtot + 0.05)
                        posiciones[j] += n_vec * (overlap * masas[i] / mtot + 0.05)

    iteracion += 1


# =============================================================================
# 5. Animación en 3D
# =============================================================================
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection='3d')

lineas_estela = []
puntos_cuerpo = []

for i in range(N):
    linea, = ax.plot([], [], [], color=colores[i], label=etiquetas[i], alpha=0.6)
    lineas_estela.append(linea)
    
    tamano_punto = 12 if masas[i] >= M_limite_fusion else 6
    punto, = ax.plot([], [], [], 'o', color=colores[i], markersize=tamano_punto, markeredgecolor='black', zorder=5)
    puntos_cuerpo.append(punto)

ax.set_xlim3d(-limite_ejes, limite_ejes)
ax.set_ylim3d(-limite_ejes, limite_ejes)
ax.set_zlim3d(-limite_ejes, limite_ejes)

ax.set_title(f"Simulación de Dispersión y Acreción (4 Cuerpos 3D)", fontsize=14, fontweight='bold')
ax.set_xlabel("Posición en x [pc]")
ax.set_ylabel("Posición en y [pc]")
ax.set_zlabel("Posición en z [pc]")
ax.legend(loc='upper right')
# grid en 3D se activa por defecto, quitamos el set_aspect('equal') porque en 3D da problemas en algunas versiones

def actualizar(frame):
    for i in range(N):
        lineas_estela[i].set_data_3d(historial_x[:frame, i], historial_y[:frame, i], historial_z[:frame, i])
        puntos_cuerpo[i].set_data_3d([historial_x[frame, i]], [historial_y[frame, i]], [historial_z[frame, i]])
        
        if np.isnan(historial_x[frame, i]):
             puntos_cuerpo[i].set_data_3d([], [], [])
             
    return lineas_estela + puntos_cuerpo

# Cambiamos blit a False porque en proyecciones 3D de Matplotlib a veces da glitches visuales
animacion = FuncAnimation(fig, actualizar, frames=n, interval=10, blit=False)

animacion.save("simulacion_4B3DBH_.mp4", writer=FFMpegWriter(fps=20, metadata=dict(artist='Me'), bitrate=1800))
print("¡Animación guardada exitosamente como simulacion_4B3DBH_.mp4!")
#plt.show()

### Simulación de 4 cuerpos con simetría radial

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter 

# =============================================================================
# 1. Parámetros de la simulación
# =============================================================================
G = 0.004498         
epsilon = 0.05       
dt = 0.005          
n = 250000            

masa_critica_bh = 4.5e7  
radio_colision = 2.0     

M_total = 4e7

R = 25 # Distancia al centro de cada vertice 

# Calculamos v_0 usando la derivación rigurosa
v_0 = np.sqrt((G * M_total / (4.0 * R)) * (1.0 + 2.0 * np.sqrt(2.0)))

# =============================================================================
# 2. Condiciones iniciales
# =============================================================================
masas = np.array([M_total, M_total, M_total, M_total])
N = len(masas)

activos = np.ones(N, dtype=bool)

#===============================================#
#PSICIONES
posiciones = np.array([
    [R, 0.0, 0.0],       # Cuerpo 1 (Derecha)
    [0.0, R, 0.0],       # Cuerpo 2 (Arriba)
    [-R, 0.0, 0.0],      # Cuerpo 3 (Izquierda)
    [0.0, -R, 0.0]       # Cuerpo 4 (Abajo)
], dtype=float)
#===============================================#
# VELOCIDADES
# Las velocidades deben ser tangenciales (perpendiculares al radio)
velocidades = np.array([
    [0.0, v_0, 0.0],     # Cuerpo 1 va pa' arriba
    [-v_0, 0.0, 0.0],    # Cuerpo 2 va pa' la izquierda
    [0.0, -v_0, 0.0],    # Cuerpo 3 va pa' abajo
    [v_0, 0.0, 0.0]      # Cuerpo 4 va pa' la derecha
], dtype=float)


# Historial para las TRES dimensiones
historial_x = np.zeros((n, N))
historial_y = np.zeros((n, N))
historial_z = np.zeros((n, N)) 

# =============================================================================
# 3. Motor Físico: Cálculo de la aceleración (Sigue intacto, ya era 3D)
# =============================================================================
def calcular_aceleraciones(pos, m):
    aceleraciones = np.zeros((N, 3))
    for i in range(N):
        if not activos[i]: continue 
        for j in range(N):
            if i != j and activos[j]: 
                dr = pos[j] - pos[i]
                norm = dr[0]**2 + dr[1]**2 + dr[2]**2
                normepsilon = norm + epsilon**2
                aceleraciones[i] += G * m[j] * dr / (normepsilon ** 1.5)  
    return aceleraciones

# =============================================================================
# 4. Integración Numérica y Detección de Colisiones
# =============================================================================
iteracion = 0
while iteracion < n:
    
    # 4.1. Revisar Colisiones
    for i in range(N):
        if not activos[i]: continue
        for j in range(i + 1, N): 
            if not activos[j]: continue
            
            dr = posiciones[j] - posiciones[i]
            distancia = np.linalg.norm(dr)
            
            if distancia < radio_colision:
                es_bh_i = masas[i] >= masa_critica_bh
                es_bh_j = masas[j] >= masa_critica_bh
                
                if es_bh_i or es_bh_j:
                    if masas[i] >= masas[j]:
                        absorbedor, absorbido = i, j
                    else:
                        absorbedor, absorbido = j, i
                        
                    m_tot = masas[absorbedor] + masas[absorbido]
                    nueva_pos = (masas[absorbedor] * posiciones[absorbedor] + masas[absorbido] * posiciones[absorbido]) / m_tot
                    nueva_vel = (masas[absorbedor] * velocidades[absorbedor] + masas[absorbido] * velocidades[absorbido]) / m_tot
                    
                    masas[absorbedor] = m_tot
                    posiciones[absorbedor] = nueva_pos
                    velocidades[absorbedor] = nueva_vel
                    
                    activos[absorbido] = False
                    masas[absorbido] = 0.0

    # 4.2. Guardar el historial EN LAS TRES DIMENSIONES
    for i in range(N):  
        if activos[i]:
            historial_x[iteracion, i] = posiciones[i, 0]
            historial_y[iteracion, i] = posiciones[i, 1]
            historial_z[iteracion, i] = posiciones[i, 2]
        else:
            historial_x[iteracion, i] = np.nan
            historial_y[iteracion, i] = np.nan
            historial_z[iteracion, i] = np.nan

    # 4.3. Calcular aceleraciones e Integrar trayectorias
    acc = calcular_aceleraciones(posiciones, masas)    

    for i in range(N):
        if activos[i]:
            posiciones[i] += velocidades[i] * dt + 0.5 * acc[i] * (dt**2) 
            velocidades[i] += acc[i] * dt  
        
    iteracion += 1

print(r"Listos los datos, renderizando animación 3D...")

# =============================================================================
# 5. Animación 3D
# =============================================================================
fig = plt.figure(figsize=(10, 10))
# Convertimos el eje a 3D
ax = fig.add_subplot(111, projection='3d')

colores = ['darkorange', 'royalblue', 'forestgreen','red']
etiquetas = ['Cuerpo 1', 'Cuerpo 2', 'Cuerpo 3','Cuerpo 4']

lineas_estela = []
puntos_cuerpo = []

for i in range(N):
    # En 3D se necesitan arreglos para inicializar. El alpha le da estilo visual.
    linea, = ax.plot([], [], [], color=colores[i], label=etiquetas[i], alpha=0.6)
    lineas_estela.append(linea)
    punto, = ax.plot([], [], [], 'o', color=colores[i], markersize=6, markeredgecolor='black', zorder=5)
    puntos_cuerpo.append(punto)

ax.set_xlim(-50, 50)
ax.set_ylim(-50, 50)
ax.set_zlim(-50, 50) 

ax.set_title("Simulación N Cuerpos con Absorción en 3D", fontsize=14, fontweight='bold')
ax.set_xlabel("Posición en x [pc]")
ax.set_ylabel("Posición en y [pc]")
ax.set_zlabel("Posición en z [pc]") 
ax.legend()

def actualizar(frame):
    for i in range(N):
        # Primero pasamos X y Y en set_data
        lineas_estela[i].set_data(historial_x[:frame, i], historial_y[:frame, i])
        # Y luego inyectamos Z con set_3d_properties
        lineas_estela[i].set_3d_properties(historial_z[:frame, i])
        
        puntos_cuerpo[i].set_data([historial_x[frame, i]], [historial_y[frame, i]])
        puntos_cuerpo[i].set_3d_properties([historial_z[frame, i]])
        
    return lineas_estela + puntos_cuerpo

animacion = FuncAnimation(fig, actualizar, frames=n, interval=10, blit=False) # Blit=False ayuda a evitar bugs gráficos en ejes 3D
animacion.save("simulacion_N4B3DSy_.mp4", writer=FFMpegWriter(fps=20, metadata=dict(artist='Me'), bitrate=1800))
print("¡Animación guardada exitosamente como simulacion_N4B3DSy_.mp4!")

### Simulación de 5 cuerpos con simetría radial

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# =============================================================================
# 1. Parámetros de la simulación
# =============================================================================
G = 0.004498         
epsilon = 0.05       
dt = 0.001          # Un dt un poco más manejable para balancear precisión
n = 100000           # Reducido para que cargue en chinga y no sature la RAM

masa_critica_bh = 4.5e7  
radio_colision = 2.0     

M_outer = 4e7       # Masa de los cuerpos externos
M_centro = 8*10**9    # Masa del cuerpo central

R = 25 # Distancia al centro de cada vértice 

# Ajuste analítico riguroso de v_0 incluyendo la masa central
v_0 = np.sqrt((G / (4.0 * R)) * (M_outer * (1.0 + 2.0 * np.sqrt(2.0)) + 4.0 * M_centro))

# =============================================================================
# 2. Conditions iniciales
# =============================================================================
masas = np.array([M_outer, M_outer, M_outer, M_outer, M_centro])
N = len(masas)

activos = np.ones(N, dtype=bool)

# POSICIONES (Agregamos el centro en la coordenada 5)
posiciones = np.array([
    [R, 0.0, 0.0],       # Cuerpo 1 (Derecha)
    [0.0, R, 0.0],       # Cuerpo 2 (Arriba)
    [-R, 0.0, 0.0],      # Cuerpo 3 (Izquierda)
    [0.0, -R, 0.0],      # Cuerpo 4 (Abajo)
    [0.0, 0.0, 0.0]       # Cuerpo 5 (Centro)
], dtype=float)

# VELOCIDADES 
velocidades = np.array([
    [0.0, v_0, 0.0],     # Cuerpo 1 va pa' arriba
    [-v_0, v_0 * .8, v_0],    # Cuerpo 2 va pa' la izquierda
    [.5 * v_0 , .5 * v_0, v_0 * .3],    # Cuerpo 3 va pa' abajo
    [v_0, v_0, v_0 * .5],     # Cuerpo 4 va pa' la derecha
    [0.0, v_0, v_0]      # Cuerpo 5 (Estaría estático en el origen)
], dtype=float)

# Historial para las TRES dimensiones
historial_x = np.zeros((n, N))
historial_y = np.zeros((n, N))
historial_z = np.zeros((n, N)) 

# =============================================================================
# 3. Motor Físico: Cálculo de la aceleración
# =============================================================================
def calcular_aceleraciones(pos, m):
    aceleraciones = np.zeros((N, 3))
    for i in range(N):
        if not activos[i]: continue 
        for j in range(N):
            if i != j and activos[j]: 
                dr = pos[j] - pos[i]
                norm = dr[0]**2 + dr[1]**2 + dr[2]**2
                normepsilon = norm + epsilon**2
                aceleraciones[i] += G * m[j] * dr / (normepsilon ** 1.5)  
    return aceleraciones

# =============================================================================
# 4. Integración Numérica y Detección de Colisiones
# =============================================================================
iteracion = 0
while iteracion < n:
    
    # 4.1. Revisar Colisiones
    for i in range(N):
        if not activos[i]: continue
        for j in range(i + 1, N): 
            if not activos[j]: continue
            
            dr = posiciones[j] - posiciones[i]
            distancia = np.linalg.norm(dr)
            
            if distancia < radio_colision:
                es_bh_i = masas[i] >= masa_critica_bh
                es_bh_j = masas[j] >= masa_critica_bh
                
                if es_bh_i or es_bh_j:
                    if masas[i] >= masas[j]:
                        absorbedor, absorbido = i, j
                    else:
                        absorbedor, absorbido = j, i
                        
                    m_tot = masas[absorbedor] + masas[absorbido]
                    nueva_pos = (masas[absorbedor] * posiciones[absorbedor] + masas[absorbido] * posiciones[absorbido]) / m_tot
                    nueva_vel = (masas[absorbedor] * velocidades[absorbedor] + masas[absorbido] * velocidades[absorbido]) / m_tot
                    
                    masas[absorbedor] = m_tot
                    posiciones[absorbedor] = nueva_pos
                    velocidades[absorbedor] = nueva_vel
                    
                    activos[absorbido] = False
                    masas[absorbido] = 0.0

    # 4.2. Guardar el historial
    for i in range(N):  
        if activos[i]:
            historial_x[iteracion, i] = posiciones[i, 0]
            historial_y[iteracion, i] = posiciones[i, 1]
            historial_z[iteracion, i] = posiciones[i, 2]
        else:
            historial_x[iteracion, i] = np.nan
            historial_y[iteracion, i] = np.nan
            historial_z[iteracion, i] = np.nan

    # 4.3. Calcular aceleraciones e Integrar trayectorias
    acc = calcular_aceleraciones(posiciones, masas)    

    for i in range(N):
        if activos[i]:
            posiciones[i] += velocidades[i] * dt + 0.5 * acc[i] * (dt**2) 
            velocidades[i] += acc[i] * dt  
        
    iteracion += 1

print(r"Listos los datos, renderizando animación 3D...")

# =============================================================================
# 5. Animación 3D
# =============================================================================
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection='3d')

# Corregido: Agregada la coma y un 5to color (purple para el centro)
colores = ['darkorange', 'royalblue', 'forestgreen', 'red', 'purple']
etiquetas = ['Cuerpo 1', 'Cuerpo 2', 'Cuerpo 3', 'Cuerpo 4', 'Centro']

lineas_estela = []
puntos_cuerpo = []

for i in range(N):
    linea, = ax.plot([], [], [], color=colores[i], label=etiquetas[i], alpha=0.6)
    lineas_estela.append(linea)
    punto, = ax.plot([], [], [], 'o', color=colores[i], markersize=6, markeredgecolor='black', zorder=5)
    puntos_cuerpo.append(punto)

ax.set_xlim(-40, 70)
ax.set_ylim(-20, 550)
ax.set_zlim(-20, 1000) 

ax.set_title("Simulación N Cuerpos con Masa Central en 3D", fontsize=14, fontweight='bold')
ax.set_xlabel("Posición en x [pc]")
ax.set_ylabel("Posición en y [pc]")
ax.set_zlabel("Posición en z [pc]") 
ax.legend()

def actualizar(frame):
    for i in range(N):
        lineas_estela[i].set_data(historial_x[:frame, i], historial_y[:frame, i])
        lineas_estela[i].set_3d_properties(historial_z[:frame, i])
        
        puntos_cuerpo[i].set_data([historial_x[frame, i]], [historial_y[frame, i]])
        puntos_cuerpo[i].set_3d_properties([historial_z[frame, i]])
        
    return lineas_estela + puntos_cuerpo

animacion = FuncAnimation(fig, actualizar, frames=n, interval=10, blit=False)

animacion.save("simulacion_N5B3DSy_.mp4", writer=FFMpegWriter(fps=20, metadata=dict(artist='Me'), bitrate=1800))
print("¡Animación guardada exitosamente como simulacion_N5B3DSy_.mp4!")

### Simulación de 4 cuerpos 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter 

# =============================================================================
# 1. Parámetros de la simulación
# =============================================================================
G = 0.004498         
epsilon = 0.05       
dt = 0.0005          
n = 150000            

masa_critica_bh = 4.5e7  
radio_colision = 2.0     

# =============================================================================
# 2. Condiciones iniciales
# =============================================================================
masas = np.array([3e7, 4e7, 5e7, 2e6])
N = len(masas)

activos = np.ones(N, dtype=bool)

#===============================================#
#PSICIONES
posiciones = np.array([
    [10, 30, 5],       # cuerpo 1
    [-20, -10, -10],   # cuerpo 2
    [10, -10, 15],      # cuerpo 3
    [10, 10, 10]       #Cuerpo 4
], dtype=float)

#===============================================#
# VELOCIDADES
velocidades = np.array([
    [0, -20, 5],       # velocidad cuerpo 1
    [15, 10, -5],      # velocidad cuerpo 2
    [-10, 5, 0],        # velocidad cuerpo 3
    [10, 0, 5]      #Velocidades cuerpo 4 
], dtype=float)



# Historial para las TRES dimensiones
historial_x = np.zeros((n, N))
historial_y = np.zeros((n, N))
historial_z = np.zeros((n, N)) 

# =============================================================================
# 3. Motor Físico: Cálculo de la aceleración (Sigue intacto, ya era 3D)
# =============================================================================
def calcular_aceleraciones(pos, m):
    aceleraciones = np.zeros((N, 3))
    for i in range(N):
        if not activos[i]: continue 
        for j in range(N):
            if i != j and activos[j]: 
                dr = pos[j] - pos[i]
                norm = dr[0]**2 + dr[1]**2 + dr[2]**2
                normepsilon = norm + epsilon**2
                aceleraciones[i] += G * m[j] * dr / (normepsilon ** 1.5)  
    return aceleraciones

# =============================================================================
# 4. Integración Numérica y Detección de Colisiones
# =============================================================================
iteracion = 0
while iteracion < n:
    
    # 4.1. Revisar Colisiones
    for i in range(N):
        if not activos[i]: continue
        for j in range(i + 1, N): 
            if not activos[j]: continue
            
            dr = posiciones[j] - posiciones[i]
            distancia = np.linalg.norm(dr)
            
            if distancia < radio_colision:
                es_bh_i = masas[i] >= masa_critica_bh
                es_bh_j = masas[j] >= masa_critica_bh
                
                if es_bh_i or es_bh_j:
                    if masas[i] >= masas[j]:
                        absorbedor, absorbido = i, j
                    else:
                        absorbedor, absorbido = j, i
                        
                    m_tot = masas[absorbedor] + masas[absorbido]
                    nueva_pos = (masas[absorbedor] * posiciones[absorbedor] + masas[absorbido] * posiciones[absorbido]) / m_tot
                    nueva_vel = (masas[absorbedor] * velocidades[absorbedor] + masas[absorbido] * velocidades[absorbido]) / m_tot
                    
                    masas[absorbedor] = m_tot
                    posiciones[absorbedor] = nueva_pos
                    velocidades[absorbedor] = nueva_vel
                    
                    activos[absorbido] = False
                    masas[absorbido] = 0.0

    # 4.2. Guardar el historial EN LAS TRES DIMENSIONES
    for i in range(N):  
        if activos[i]:
            historial_x[iteracion, i] = posiciones[i, 0]
            historial_y[iteracion, i] = posiciones[i, 1]
            historial_z[iteracion, i] = posiciones[i, 2]
        else:
            historial_x[iteracion, i] = np.nan
            historial_y[iteracion, i] = np.nan
            historial_z[iteracion, i] = np.nan

    # 4.3. Calcular aceleraciones e Integrar trayectorias
    acc = calcular_aceleraciones(posiciones, masas)    

    for i in range(N):
        if activos[i]:
            posiciones[i] += velocidades[i] * dt + 0.5 * acc[i] * (dt**2) 
            velocidades[i] += acc[i] * dt  
        
    iteracion += 1

print(r"Listos los datos, renderizando animación 3D...")

# =============================================================================
# 5. Animación 3D
# =============================================================================
fig = plt.figure(figsize=(10, 10))
# Convertimos el eje a 3D
ax = fig.add_subplot(111, projection='3d')

colores = ['darkorange', 'royalblue', 'forestgreen','red']
etiquetas = ['Cuerpo 1', 'Cuerpo 2', 'Cuerpo 3','Cuerpo 4']

lineas_estela = []
puntos_cuerpo = []

for i in range(N):
    # En 3D se necesitan arreglos para inicializar. El alpha le da estilo visual.
    linea, = ax.plot([], [], [], color=colores[i], label=etiquetas[i], alpha=0.6)
    lineas_estela.append(linea)
    punto, = ax.plot([], [], [], 'o', color=colores[i], markersize=6, markeredgecolor='black', zorder=5)
    puntos_cuerpo.append(punto)

ax.set_xlim(-50, 50)
ax.set_ylim(-50, 50)
ax.set_zlim(-50, 50) 

ax.set_title("Simulación N Cuerpos con Absorción en 3D", fontsize=14, fontweight='bold')
ax.set_xlabel("Posición en x [pc]")
ax.set_ylabel("Posición en y [pc]")
ax.set_zlabel("Posición en z [pc]") 
ax.legend()

def actualizar(frame):
    for i in range(N):
        # Primero pasamos X y Y en set_data
        lineas_estela[i].set_data(historial_x[:frame, i], historial_y[:frame, i])
        # Y luego inyectamos Z con set_3d_properties
        lineas_estela[i].set_3d_properties(historial_z[:frame, i])
        
        puntos_cuerpo[i].set_data([historial_x[frame, i]], [historial_y[frame, i]])
        puntos_cuerpo[i].set_3d_properties([historial_z[frame, i]])
        
    return lineas_estela + puntos_cuerpo

animacion = FuncAnimation(fig, actualizar, frames=n, interval=10, blit=False) # Blit=False ayuda a evitar bugs gráficos en ejes 3D

animacion.save("simulacion_B4D3D_.mp4", writer=FFMpegWriter(fps=20, metadata=dict(artist='Me'), bitrate=1800))
print("¡Animación guardada exitosamente como simulacion_B4D3D_.mp4!")

### Simulación de 3 cuerpos con simetría radial

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# =============================================================================
# 1. Parámetros de la simulación
# =============================================================================
G =  0.004498         
epsilon = 0.005       
dt = 0.005          
n = 150000           

masa_critica_bh = 4.5e7  
radio_colision = 2.0     

# =============================================================================
# 2. Condiciones iniciales
# =============================================================================
masas = np.array([10**5, 10, 10])
N = len(masas)

activos = np.ones(N, dtype=bool)


#===============================================#
#PSICIONES
posiciones = np.array([
    [10, 0, 0.0],  # Cuerpo 1
    [0, 10, 0.0],  # Cuerpo 2
    [0.0, 0.0, 0.0]    # Cuerpo 3 
], dtype=float)

#===============================================#
# VELOCIDADES
velocidades = np.array([
    [0, 0, 0.0],
    [0,0 , 0.0],
    [0 , 0, 0.0]
], dtype=float)



# Historial para las TRES dimensiones
historial_x = np.zeros((n, N))
historial_y = np.zeros((n, N))
historial_z = np.zeros((n, N)) 

# =============================================================================
# 3. Motor Físico: Cálculo de la aceleración (Sigue intacto, ya era 3D)
# =============================================================================
def calcular_aceleraciones(pos, m):
    aceleraciones = np.zeros((N, 3))
    for i in range(N):
        if not activos[i]: continue 
        for j in range(N):
            if i != j and activos[j]: 
                dr = pos[j] - pos[i]
                norm = dr[0]**2 + dr[1]**2 + dr[2]**2
                normepsilon = norm + epsilon**2
                aceleraciones[i] += G * m[j] * dr / (normepsilon ** 1.5)  
    return aceleraciones

# =============================================================================
# 4. Integración Numérica y Detección de Colisiones
# =============================================================================
iteracion = 0
while iteracion < n:
    
    # 4.1. Revisar Colisiones
    for i in range(N):
        if not activos[i]: continue
        for j in range(i + 1, N): 
            if not activos[j]: continue
            
            dr = posiciones[j] - posiciones[i]
            distancia = np.linalg.norm(dr)
            
            if distancia < radio_colision:
                es_bh_i = masas[i] >= masa_critica_bh
                es_bh_j = masas[j] >= masa_critica_bh
                
                if es_bh_i or es_bh_j:
                    if masas[i] >= masas[j]:
                        absorbedor, absorbido = i, j
                    else:
                        absorbedor, absorbido = j, i
                        
                    m_tot = masas[absorbedor] + masas[absorbido]
                    nueva_pos = (masas[absorbedor] * posiciones[absorbedor] + masas[absorbido] * posiciones[absorbido]) / m_tot
                    nueva_vel = (masas[absorbedor] * velocidades[absorbedor] + masas[absorbido] * velocidades[absorbido]) / m_tot
                    
                    masas[absorbedor] = m_tot
                    posiciones[absorbedor] = nueva_pos
                    velocidades[absorbedor] = nueva_vel
                    
                    activos[absorbido] = False
                    masas[absorbido] = 0.0

    # 4.2. Guardar el historial EN LAS TRES DIMENSIONES
    for i in range(N):  
        if activos[i]:
            historial_x[iteracion, i] = posiciones[i, 0]
            historial_y[iteracion, i] = posiciones[i, 1]
            historial_z[iteracion, i] = posiciones[i, 2]
        else:
            historial_x[iteracion, i] = np.nan
            historial_y[iteracion, i] = np.nan
            historial_z[iteracion, i] = np.nan

    # 4.3. Calcular aceleraciones e Integrar trayectorias
    acc = calcular_aceleraciones(posiciones, masas)    

    for i in range(N):
        if activos[i]:
            posiciones[i] += velocidades[i] * dt + 0.5 * acc[i] * (dt**2) 
            velocidades[i] += acc[i] * dt  
        
    iteracion += 1

print(r"Listos los datos, renderizando animación 3D...")

# =============================================================================
# 5. Animación 3D
# =============================================================================
fig = plt.figure(figsize=(10, 10))
# Convertimos el eje a 3D
ax = fig.add_subplot(111, projection='3d')

colores = ['darkorange', 'royalblue','red']
etiquetas = ['Cuerpo 1', 'Cuerpo 2', 'Cuerpo 3']

lineas_estela = []
puntos_cuerpo = []

for i in range(N):
    # En 3D se necesitan arreglos para inicializar. El alpha le da estilo visual.
    linea, = ax.plot([], [], [], color=colores[i], label=etiquetas[i], alpha=0.6)
    lineas_estela.append(linea)
    punto, = ax.plot([], [], [], 'o', color=colores[i], markersize=6, markeredgecolor='black', zorder=5)
    puntos_cuerpo.append(punto)

ax.set_xlim(-30, 30)
ax.set_ylim(-30, 30)
ax.set_zlim(-30, 30) 

ax.set_title("Simulación N Cuerpos con Absorción en 3D", fontsize=14, fontweight='bold')
ax.set_xlabel("Posición en x [pc]")
ax.set_ylabel("Posición en y [pc]")
ax.set_zlabel("Posición en z [pc]") 
ax.legend()

def actualizar(frame):
    for i in range(N):
        # Primero pasamos X y Y en set_data
        lineas_estela[i].set_data(historial_x[:frame, i], historial_y[:frame, i])
        # Y luego inyectamos Z con set_3d_properties
        lineas_estela[i].set_3d_properties(historial_z[:frame, i])
        
        puntos_cuerpo[i].set_data([historial_x[frame, i]], [historial_y[frame, i]])
        puntos_cuerpo[i].set_3d_properties([historial_z[frame, i]])
        
    return lineas_estela + puntos_cuerpo

animacion = FuncAnimation(fig, actualizar, frames=n, interval=10, blit=False) # Blit=False ayuda a evitar bugs gráficos en ejes 3D

animacion.save("simulacion_N3B3DSy.mp4", writer=FFMpegWriter(fps=20, metadata=dict(artist='Me'), bitrate=1800))
print("¡Animación guardada exitosamente como simulacion_N3B3DSy_.mp4!")

### Simulación del problema de Sitnikov

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# =============================================================================
# 1. Parámetros de la simulación
# =============================================================================
G =  0.004498         
epsilon = 0.005       
dt = 0.005          
n = 150000           

masa_critica_bh = 4.5e7  
radio_colision = 2.0     

M_primaria = 4e7
m_prueba = 100.0  # Masa casi despreciable para no perturbar a las grandes
masas = np.array([M_primaria, M_primaria, m_prueba])
# Radio de la órbita de las masas primarias en el plano XY
R = 20.0
# Altura inicial del cuerpo de prueba en el eje Z
Z_0 = 25.0
# Calculamos la velocidad orbital analíticamente para órbitas circulares
# Usamos tu misma G = 0.004498
v_0 = np.sqrt((G * M_primaria) / (4 * R))



# =============================================================================
# 2. Condiciones iniciales
# =============================================================================
#masas = np.array([10**5, 10, 10])
N = len(masas)

activos = np.ones(N, dtype=bool)


#===============================================#
#PSICIONES
posiciones = np.array([
    [R, 0, 0.0],  # Cuerpo 1
    [-R, 10, 0.0],  # Cuerpo 2
    [0.0, 0.0, Z_0]    # Cuerpo 3 
], dtype=float)

#===============================================#
# VELOCIDADES
velocidades = np.array([
    [0, v_0, 0.0],
    [0.0 , -v_0 , 0.0],
    [0 , 0, 0.0]
], dtype=float)



# Historial para las TRES dimensiones
historial_x = np.zeros((n, N))
historial_y = np.zeros((n, N))
historial_z = np.zeros((n, N)) 

# =============================================================================
# 3. Motor Físico: Cálculo de la aceleración (Sigue intacto, ya era 3D)
# =============================================================================
def calcular_aceleraciones(pos, m):
    aceleraciones = np.zeros((N, 3))
    for i in range(N):
        if not activos[i]: continue 
        for j in range(N):
            if i != j and activos[j]: 
                dr = pos[j] - pos[i]
                norm = dr[0]**2 + dr[1]**2 + dr[2]**2
                normepsilon = norm + epsilon**2
                aceleraciones[i] += G * m[j] * dr / (normepsilon ** 1.5)  
    return aceleraciones

# =============================================================================
# 4. Integración Numérica y Detección de Colisiones
# =============================================================================
iteracion = 0
while iteracion < n:
    
    # 4.1. Revisar Colisiones
    for i in range(N):
        if not activos[i]: continue
        for j in range(i + 1, N): 
            if not activos[j]: continue
            
            dr = posiciones[j] - posiciones[i]
            distancia = np.linalg.norm(dr)
            
            if distancia < radio_colision:
                es_bh_i = masas[i] >= masa_critica_bh
                es_bh_j = masas[j] >= masa_critica_bh
                
                if es_bh_i or es_bh_j:
                    if masas[i] >= masas[j]:
                        absorbedor, absorbido = i, j
                    else:
                        absorbedor, absorbido = j, i
                        
                    m_tot = masas[absorbedor] + masas[absorbido]
                    nueva_pos = (masas[absorbedor] * posiciones[absorbedor] + masas[absorbido] * posiciones[absorbido]) / m_tot
                    nueva_vel = (masas[absorbedor] * velocidades[absorbedor] + masas[absorbido] * velocidades[absorbido]) / m_tot
                    
                    masas[absorbedor] = m_tot
                    posiciones[absorbedor] = nueva_pos
                    velocidades[absorbedor] = nueva_vel
                    
                    activos[absorbido] = False
                    masas[absorbido] = 0.0

    # 4.2. Guardar el historial EN LAS TRES DIMENSIONES
    for i in range(N):  
        if activos[i]:
            historial_x[iteracion, i] = posiciones[i, 0]
            historial_y[iteracion, i] = posiciones[i, 1]
            historial_z[iteracion, i] = posiciones[i, 2]
        else:
            historial_x[iteracion, i] = np.nan
            historial_y[iteracion, i] = np.nan
            historial_z[iteracion, i] = np.nan

    # 4.3. Calcular aceleraciones e Integrar trayectorias
    acc = calcular_aceleraciones(posiciones, masas)    

    for i in range(N):
        if activos[i]:
            posiciones[i] += velocidades[i] * dt + 0.5 * acc[i] * (dt**2) 
            velocidades[i] += acc[i] * dt  
        
    iteracion += 1

print(r"Listos los datos, renderizando animación 3D...")

# =============================================================================
# 5. Animación 3D
# =============================================================================
fig = plt.figure(figsize=(10, 10))
# Convertimos el eje a 3D
ax = fig.add_subplot(111, projection='3d')

colores = ['darkorange', 'royalblue','red']
etiquetas = ['Cuerpo 1', 'Cuerpo 2', 'Cuerpo 3']

lineas_estela = []
puntos_cuerpo = []

for i in range(N):
    # En 3D se necesitan arreglos para inicializar. El alpha le da estilo visual.
    linea, = ax.plot([], [], [], color=colores[i], label=etiquetas[i], alpha=0.6)
    lineas_estela.append(linea)
    punto, = ax.plot([], [], [], 'o', color=colores[i], markersize=6, markeredgecolor='black', zorder=5)
    puntos_cuerpo.append(punto)

ax.set_xlim(-30, 30)
ax.set_ylim(-30, 30)
ax.set_zlim(-30, 30) 

ax.set_title("Simulación N Cuerpos con Absorción en 3D", fontsize=14, fontweight='bold')
ax.set_xlabel("Posición en x [pc]")
ax.set_ylabel("Posición en y [pc]")
ax.set_zlabel("Posición en z [pc]") 
ax.legend()

def actualizar(frame):
    for i in range(N):
        # Primero pasamos X y Y en set_data
        lineas_estela[i].set_data(historial_x[:frame, i], historial_y[:frame, i])
        # Y luego inyectamos Z con set_3d_properties
        lineas_estela[i].set_3d_properties(historial_z[:frame, i])
        
        puntos_cuerpo[i].set_data([historial_x[frame, i]], [historial_y[frame, i]])
        puntos_cuerpo[i].set_3d_properties([historial_z[frame, i]])
        
    return lineas_estela + puntos_cuerpo

animacion = FuncAnimation(fig, actualizar, frames=n, interval=10, blit=False) # Blit=False ayuda a evitar bugs gráficos en ejes 3D

animacion.save("simulacion_N3Bsitnikov.mp4", writer=FFMpegWriter(fps=20, metadata=dict(artist='Me'), bitrate=1800))
print("¡Animación guardada exitosamente como simulacion_N3Bsitnikov_.mp4!")

### Simulación de 3 cuerpos con absorbción

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# =============================================================================
# 1. Parámetros de la simulación
# =============================================================================
G = 1 #0.004498         
epsilon = 1e-10       
dt = 0.005          
n = 150000           

masa_critica_bh = 4.5e7  
radio_colision = 2.0     

# =============================================================================
# 2. Condiciones iniciales
# =============================================================================
masas = np.array([10, 10, 10])
N = len(masas)

activos = np.ones(N, dtype=bool)

#================================================#
# Las coordenadas mágicas de Chenciner
v_x = 0.4662036850
v_y = 0.4323657300
x_0 = 0.97000436
y_0 = 0.24308753

#===============================================#
#PSICIONES
posiciones = np.array([
    [x_0, -y_0, 0.0],  # Cuerpo 1
    [-x_0, y_0, 0.0],  # Cuerpo 2
    [0.0, 0.0, 0.0]    # Cuerpo 3 
], dtype=float)

#===============================================#
# VELOCIDADES
velocidades = np.array([
    [v_x, v_y, 0.0],
    [v_x, v_y, 0.0],
    [-2 * v_x, -2 * v_y, 0.0]
], dtype=float)



# Historial para las TRES dimensiones
historial_x = np.zeros((n, N))
historial_y = np.zeros((n, N))
historial_z = np.zeros((n, N)) 

# =============================================================================
# 3. Motor Físico: Cálculo de la aceleración (Sigue intacto, ya era 3D)
# =============================================================================
def calcular_aceleraciones(pos, m):
    aceleraciones = np.zeros((N, 3))
    for i in range(N):
        if not activos[i]: continue 
        for j in range(N):
            if i != j and activos[j]: 
                dr = pos[j] - pos[i]
                norm = dr[0]**2 + dr[1]**2 + dr[2]**2
                normepsilon = norm + epsilon**2
                aceleraciones[i] += G * m[j] * dr / (normepsilon ** 1.5)  
    return aceleraciones

# =============================================================================
# 4. Integración Numérica y Detección de Colisiones
# =============================================================================
iteracion = 0
while iteracion < n:
    
    # 4.1. Revisar Colisiones
    for i in range(N):
        if not activos[i]: continue
        for j in range(i + 1, N): 
            if not activos[j]: continue
            
            dr = posiciones[j] - posiciones[i]
            distancia = np.linalg.norm(dr)
            
            if distancia < radio_colision:
                es_bh_i = masas[i] >= masa_critica_bh
                es_bh_j = masas[j] >= masa_critica_bh
                
                if es_bh_i or es_bh_j:
                    if masas[i] >= masas[j]:
                        absorbedor, absorbido = i, j
                    else:
                        absorbedor, absorbido = j, i
                        
                    m_tot = masas[absorbedor] + masas[absorbido]
                    nueva_pos = (masas[absorbedor] * posiciones[absorbedor] + masas[absorbido] * posiciones[absorbido]) / m_tot
                    nueva_vel = (masas[absorbedor] * velocidades[absorbedor] + masas[absorbido] * velocidades[absorbido]) / m_tot
                    
                    masas[absorbedor] = m_tot
                    posiciones[absorbedor] = nueva_pos
                    velocidades[absorbedor] = nueva_vel
                    
                    activos[absorbido] = False
                    masas[absorbido] = 0.0

    # 4.2. Guardar el historial EN LAS TRES DIMENSIONES
    for i in range(N):  
        if activos[i]:
            historial_x[iteracion, i] = posiciones[i, 0]
            historial_y[iteracion, i] = posiciones[i, 1]
            historial_z[iteracion, i] = posiciones[i, 2]
        else:
            historial_x[iteracion, i] = np.nan
            historial_y[iteracion, i] = np.nan
            historial_z[iteracion, i] = np.nan

    # 4.3. Calcular aceleraciones e Integrar trayectorias
    acc = calcular_aceleraciones(posiciones, masas)    

    for i in range(N):
        if activos[i]:
            posiciones[i] += velocidades[i] * dt + 0.5 * acc[i] * (dt**2) 
            velocidades[i] += acc[i] * dt  
        
    iteracion += 1

print(r"Listos los datos, renderizando animación 3D...")

# =============================================================================
# 5. Animación 3D
# =============================================================================
fig = plt.figure(figsize=(10, 10))
# Convertimos el eje a 3D
ax = fig.add_subplot(111, projection='3d')

colores = ['darkorange', 'royalblue','red']
etiquetas = ['Cuerpo 1', 'Cuerpo 2', 'Cuerpo 3']

lineas_estela = []
puntos_cuerpo = []

for i in range(N):
    # En 3D se necesitan arreglos para inicializar. El alpha le da estilo visual.
    linea, = ax.plot([], [], [], color=colores[i], label=etiquetas[i], alpha=0.6)
    lineas_estela.append(linea)
    punto, = ax.plot([], [], [], 'o', color=colores[i], markersize=6, markeredgecolor='black', zorder=5)
    puntos_cuerpo.append(punto)

ax.set_xlim(-2, 2)
ax.set_ylim(-2, 2)
ax.set_zlim(-2, 2) 

ax.set_title("Simulación N Cuerpos con Absorción en 3D", fontsize=14, fontweight='bold')
ax.set_xlabel("Posición en x [pc]")
ax.set_ylabel("Posición en y [pc]")
ax.set_zlabel("Posición en z [pc]") 
ax.legend()

def actualizar(frame):
    for i in range(N):
        # Primero pasamos X y Y en set_data
        lineas_estela[i].set_data(historial_x[:frame, i], historial_y[:frame, i])
        # Y luego inyectamos Z con set_3d_properties
        lineas_estela[i].set_3d_properties(historial_z[:frame, i])
        
        puntos_cuerpo[i].set_data([historial_x[frame, i]], [historial_y[frame, i]])
        puntos_cuerpo[i].set_3d_properties([historial_z[frame, i]])
        
    return lineas_estela + puntos_cuerpo

animacion = FuncAnimation(fig, actualizar, frames=n, interval=10, blit=False) # Blit=False ayuda a evitar bugs gráficos en ejes 3D

animacion.save("simulacion_N3B3D.mp4", writer=FFMpegWriter(fps=20, metadata=dict(artist='Me'), bitrate=1800))
print("¡Animación guardada exitosamente como simulacion_N3B3D_.mp4!")

### Simulación de un sistema binario con la inclusión de un tercer cuerpo

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# =============================================================================
# 1. Parámetros de la simulación
# =============================================================================
G =  0.004498         
epsilon = 0.005       
dt = 0.0005          
n = 150000           

masa_critica_bh = 4.5e7  
radio_colision = 2.0     

M_base = 4e7
R_bin = 10.0

v_bin = np.sqrt((G * M_base ) / ( 4*R_bin ))

angulos = np.array([0.0, 2.0 * np.pi / 3.0, 4.0 * np.pi / 3.0])

D_extra = 40.0
V_extra = -50


# =============================================================================
# 2. Condiciones iniciales
# =============================================================================
masas = np.array([M_base, M_base, M_base])
N = len(masas)

activos = np.ones(N, dtype=bool)


#===============================================#
#PSICIONES
posiciones = np.array([
    [R_bin, 0, 0],  # Cuerpo 1
    [R_bin, 0, 0],  # Cuerpo 2
    [0, D_extra, 0]    # Cuerpo 3 
], dtype=float)

#===============================================#
# VELOCIDADES
velocidades = np.array([
    [0, v_bin, 0],
    [0, -v_bin, 0],
    [0, V_extra, 0.0]
], dtype=float)



# Historial para las TRES dimensiones
historial_x = np.zeros((n, N))
historial_y = np.zeros((n, N))
historial_z = np.zeros((n, N)) 

# =============================================================================
# 3. Motor Físico: Cálculo de la aceleración (Sigue intacto, ya era 3D)
# =============================================================================
def calcular_aceleraciones(pos, m):
    aceleraciones = np.zeros((N, 3))
    for i in range(N):
        if not activos[i]: continue 
        for j in range(N):
            if i != j and activos[j]: 
                dr = pos[j] - pos[i]
                norm = dr[0]**2 + dr[1]**2 + dr[2]**2
                normepsilon = norm + epsilon**2
                aceleraciones[i] += G * m[j] * dr / (normepsilon ** 1.5)  
    return aceleraciones

# =============================================================================
# 4. Integración Numérica y Detección de Colisiones
# =============================================================================
iteracion = 0
while iteracion < n:
    
    # 4.1. Revisar Colisiones
    for i in range(N):
        if not activos[i]: continue
        for j in range(i + 1, N): 
            if not activos[j]: continue
            
            dr = posiciones[j] - posiciones[i]
            distancia = np.linalg.norm(dr)
            
            if distancia < radio_colision:
                es_bh_i = masas[i] >= masa_critica_bh
                es_bh_j = masas[j] >= masa_critica_bh
                
                if es_bh_i or es_bh_j:
                    if masas[i] >= masas[j]:
                        absorbedor, absorbido = i, j
                    else:
                        absorbedor, absorbido = j, i
                        
                    m_tot = masas[absorbedor] + masas[absorbido]
                    nueva_pos = (masas[absorbedor] * posiciones[absorbedor] + masas[absorbido] * posiciones[absorbido]) / m_tot
                    nueva_vel = (masas[absorbedor] * velocidades[absorbedor] + masas[absorbido] * velocidades[absorbido]) / m_tot
                    
                    masas[absorbedor] = m_tot
                    posiciones[absorbedor] = nueva_pos
                    velocidades[absorbedor] = nueva_vel
                    
                    activos[absorbido] = False
                    masas[absorbido] = 0.0

    # 4.2. Guardar el historial EN LAS TRES DIMENSIONES
    for i in range(N):  
        if activos[i]:
            historial_x[iteracion, i] = posiciones[i, 0]
            historial_y[iteracion, i] = posiciones[i, 1]
            historial_z[iteracion, i] = posiciones[i, 2]
        else:
            historial_x[iteracion, i] = np.nan
            historial_y[iteracion, i] = np.nan
            historial_z[iteracion, i] = np.nan

    # 4.3. Calcular aceleraciones e Integrar trayectorias
    acc = calcular_aceleraciones(posiciones, masas)    

    for i in range(N):
        if activos[i]:
            posiciones[i] += velocidades[i] * dt + 0.5 * acc[i] * (dt**2) 
            velocidades[i] += acc[i] * dt  
        
    iteracion += 1

print(r"Listos los datos, renderizando animación 3D...")

# =============================================================================
# 5. Animación 3D
# =============================================================================
fig = plt.figure(figsize=(10, 10))
# Convertimos el eje a 3D
ax = fig.add_subplot(111, projection='3d')

colores = ['darkorange', 'royalblue','red']
etiquetas = ['Cuerpo 1', 'Cuerpo 2', 'Cuerpo 3']

lineas_estela = []
puntos_cuerpo = []

for i in range(N):
    # En 3D se necesitan arreglos para inicializar. El alpha le da estilo visual.
    linea, = ax.plot([], [], [], color=colores[i], label=etiquetas[i], alpha=0.6)
    lineas_estela.append(linea)
    punto, = ax.plot([], [], [], 'o', color=colores[i], markersize=6, markeredgecolor='black', zorder=5)
    puntos_cuerpo.append(punto)

ax.set_xlim(-50, 50)
ax.set_ylim(-50, 50)
ax.set_zlim(-50, 50) 

ax.set_title("Simulación N Cuerpos con Absorción en 3D", fontsize=14, fontweight='bold')
ax.set_xlabel("Posición en x [pc]")
ax.set_ylabel("Posición en y [pc]")
ax.set_zlabel("Posición en z [pc]") 
ax.legend()

def actualizar(frame):
    for i in range(N):
        # Primero pasamos X y Y en set_data
        lineas_estela[i].set_data(historial_x[:frame, i], historial_y[:frame, i])
        # Y luego inyectamos Z con set_3d_properties
        lineas_estela[i].set_3d_properties(historial_z[:frame, i])
        
        puntos_cuerpo[i].set_data([historial_x[frame, i]], [historial_y[frame, i]])
        puntos_cuerpo[i].set_3d_properties([historial_z[frame, i]])
        
    return lineas_estela + puntos_cuerpo

animacion = FuncAnimation(fig, actualizar, frames=n, interval=10, blit=False) # Blit=False ayuda a evitar bugs gráficos en ejes 3D

animacion.save("simulacion_N3B3Dbin.mp4", writer=FFMpegWriter(fps=20, metadata=dict(artist='Me'), bitrate=1800))
print("¡Animación guardada exitosamente como simulacion_N3B3Dbin_.mp4!")
plt.show()